# Lab: LoRA from scratch

**Session 3: Post-training (SFT, RLHF and DPO)**

Freeze a model's weights and train a small low-rank update next to them:

$$h = W_0\,x + \frac{\alpha}{r}\, B A\, x \qquad W_0 \text{ frozen}, \quad A \in \mathbb{R}^{r \times d_{in}},\ B \in \mathbb{R}^{d_{out} \times r} \text{ trained}$$

You complete **two cells**, marked **Your turn**. Every other cell is marked **Given**: read it, then run it.

1. Write the LoRA layer (your turn)
2. Add it to a small instruct model (given)
3. Fine-tune on OpenAssistant answers (your turn: the training step)
4. Compare the model with and without the adapter (given)


## Setup (given)

Runs on a laptop with `Qwen2.5-0.5B-Instruct` (no account, no API key). Training takes a few minutes on a laptop, less on a GPU (Google Colab works).


In [ ]:
# %pip install -q torch transformers datasets matplotlib peft

In [ ]:
import time

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"  # Apple Silicon
else:
    DEVICE = "cpu"
torch.manual_seed(0)

tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).to(DEVICE)
n_total = sum(p.numel() for p in model.parameters())
print(f"{MODEL}: {n_total / 1e6:.0f}M parameters on {DEVICE}")

## Part 1: the LoRA layer (your turn)

`LoRALinear` wraps an existing `nn.Linear`, called `self.base`, which stays frozen. Next to it you add two small layers, both `nn.Linear` with `bias=False` (a linear layer without bias is just a matrix):

- `self.A`: from `in_features` down to `r`. Its default random initialisation is fine.
- `self.B`: from `r` back up to `out_features`. The next line (given) sets its weights to **zero**, so at step 0 the model is exactly the original one.
- in `forward`: add the LoRA term `B(A(dropout(x)))`, multiplied by `self.scaling` (= alpha / r).

Three lines to write, marked `# Your turn`.


In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base = base.requires_grad_(False)  # W0 is frozen
        self.scaling = alpha / r
        self.enabled = True  # switch the adapter off and on (Part 4)
        self.dropout = nn.Dropout(dropout)
        kw = {"device": base.weight.device, "dtype": base.weight.dtype}

        self.A = ...  # Your turn: nn.Linear(in_features -> r, no bias, **kw)
        self.B = ...  # Your turn: nn.Linear(r -> out_features, no bias, **kw)
        nn.init.zeros_(self.B.weight)  # given: B = 0, so step 0 = original model

    def forward(self, x):
        out = self.base(x)
        if self.enabled:
            out = out + ...  # Your turn: B(A(dropout(x))), times self.scaling
        return out

    @torch.no_grad()
    def merged_weight(self):  # W = W0 + (alpha / r) BA
        return self.base.weight + self.scaling * self.B.weight @ self.A.weight

Checks (given). If they all pass, your layer is correct.


In [ ]:
layer = nn.Linear(64, 32)
lora = LoRALinear(layer, r=4, alpha=8)
x = torch.randn(5, 64)

# 0. A goes from 64 down to r = 4, B from 4 back up to 32
assert isinstance(lora.A, nn.Linear) and isinstance(lora.B, nn.Linear), (
    "Part 1: define self.A and self.B with nn.Linear"
)
assert lora.A.weight.shape == (4, 64), "A: nn.Linear(base.in_features, r, ...)"
assert lora.B.weight.shape == (32, 4), "B: nn.Linear(r, base.out_features, ...)"

# 1. B = 0 at the start: same output as the original layer
assert torch.allclose(lora(x), layer(x)), "B's weight must start at zero"

# 2. Only A and B are trainable: r x (in + out) parameters, no bias
trainable = sum(p.numel() for p in lora.parameters() if p.requires_grad)
assert trainable == 4 * (64 + 32), f"expected 384 trainable, got {trainable} (bias=False?)"

# 3. Gradients reach A and B, never the frozen weight
lora(x).sum().backward()
assert layer.weight.grad is None and lora.B.weight.grad is not None

# 4. After training, BA merges into W: same output, no extra cost
with torch.no_grad():
    lora.B.weight.normal_()
lora.eval()  # no dropout
merged = x @ lora.merged_weight().T + layer.bias
assert torch.allclose(lora(x), merged, atol=1e-5), "check the forward pass"
print(f"OK: {trainable} trainable parameters instead of {64 * 32 + 32}")

## Part 2: add LoRA to the model (given)

`add_lora` freezes the whole model, then wraps the attention **query** and **value** projections (`q_proj`, `v_proj`) of every layer in a `LoRALinear`. Because `B` starts at zero, the model gives exactly the same outputs as before.


In [ ]:
def add_lora(model, targets=("q_proj", "v_proj"), r=8, alpha=16):
    model.requires_grad_(False)
    for name, module in list(model.named_modules()):
        parent, _, child = name.rpartition(".")
        if isinstance(module, nn.Linear) and child in targets:
            setattr(model.get_submodule(parent), child, LoRALinear(module, r, alpha))
    for module in model.modules():  # A and B stay trainable
        if isinstance(module, LoRALinear):
            module.A.requires_grad_(True)
            module.B.requires_grad_(True)
    return model


probe = tok("The European Central Bank", return_tensors="pt").to(DEVICE)
with torch.no_grad():
    before = model(**probe).logits
    model = add_lora(model).eval()
    after = model(**probe).logits
assert torch.allclose(before, after, atol=1e-5), "model changed at initialisation"

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_layers = sum(isinstance(m, LoRALinear) for m in model.modules())
print(f"{n_layers} LoRA layers, {n_trainable:,} trainable parameters")
print(f"= {n_trainable / n_total:.2%} of the model; outputs unchanged")

## Part 3: fine-tune on OpenAssistant

**Data (given).** Each example is the first user message of an English OpenAssistant conversation and the answer that volunteers ranked **best**. The prompt tokens get the label `-100`, which the loss ignores: the model learns to **answer**, not to ask.


In [ ]:
N_TRAIN, N_VALID, MAX_LEN = 200, 24, 256

oasst = load_dataset("OpenAssistant/oasst1")
messages = {m["message_id"]: m for split in oasst.values() for m in split}


def is_first_best_answer(m):
    parent = messages.get(m["parent_id"])
    return (
        m["role"] == "assistant"
        and m["rank"] == 0
        and m["lang"] == "en"
        and parent is not None
        and parent["parent_id"] is None
    )


def to_example(m):
    chat = [{"role": "user", "content": messages[m["parent_id"]]["text"]}]
    reply = [{"role": "assistant", "content": m["text"]}]
    ids = lambda c, **kw: tok.apply_chat_template(c, return_dict=True, **kw)[
        "input_ids"
    ]
    prompt, full = ids(chat, add_generation_prompt=True), ids(chat + reply)
    labels = [-100] * len(prompt) + full[len(prompt) :]  # loss on the answer only
    return {
        "input_ids": torch.tensor([full[:MAX_LEN]], device=DEVICE),
        "labels": torch.tensor([labels[:MAX_LEN]], device=DEVICE),
    }


def load_split(split, n):
    rows = oasst[split].filter(is_first_best_answer).shuffle(seed=0).select(range(n))
    return [to_example(m) for m in rows]


train, valid = load_split("train", N_TRAIN), load_split("validation", N_VALID)
print(f"{len(train)} training and {len(valid)} validation examples")

**Training step (your turn).** One example per step. The optimiser only receives `A` and `B`, with the LoRA learning rate from the slides (2e-4). When you pass `labels`, the model computes the shifted next-token loss itself: `model(**batch).loss`.

Write the four lines of a training step: forward pass, backward pass, optimiser step, reset the gradients.


In [ ]:
EVAL_EVERY = 25
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=2e-4
)


@torch.no_grad()
def validation_loss():
    model.eval()
    losses = [model(**example).loss.item() for example in valid]
    model.train()
    return sum(losses) / len(losses)


history = {"train": [], "valid": [(0, validation_loss())]}
print(f"step   0: validation loss {history['valid'][0][1]:.3f}")
model.train()
start = time.perf_counter()
for step, batch in enumerate(train, start=1):
    # Your turn: forward pass, backward pass, optimiser step, reset gradients
    loss = ...

    history["train"].append((step, loss.item()))
    if step % EVAL_EVERY == 0:
        history["valid"].append((step, validation_loss()))
        print(f"step {step:>3}: validation loss {history['valid'][-1][1]:.3f}")
print(f"training time: {(time.perf_counter() - start) / 60:.1f} min")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(*zip(*history["train"]), alpha=0.4, label="train (one example)")
ax.plot(*zip(*history["valid"]), marker="o", linewidth=2, label="validation (held-out)")
ax.set_xlabel("step")
ax.set_ylabel("loss on answer tokens")
ax.legend()
fig.savefig("loss.png", dpi=150, bbox_inches="tight")
plt.show()

## Part 4: adapter off vs on (given)

The `enabled` flag gives back the **original** model for free. Same prompts, greedy decoding: any difference comes from the adapter. Then save the adapter: only `A` and `B`.


In [ ]:
def set_lora(model, enabled):
    for module in model.modules():
        if isinstance(module, LoRALinear):
            module.enabled = enabled


@torch.no_grad()
def answer(question, max_new_tokens=120):
    model.eval()
    chat = [{"role": "user", "content": question}]
    inputs = tok.apply_chat_template(
        chat, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(DEVICE)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0, inputs["input_ids"].shape[1] :], skip_special_tokens=True)


PROMPTS = [
    "How do I politely decline a meeting invitation?",
    "Explain what inflation is to a 12-year-old.",
]
for q in PROMPTS:
    set_lora(model, False)
    original = answer(q)
    set_lora(model, True)
    tuned = answer(q)
    print(f"QUESTION: {q}\n\nORIGINAL:\n{original}\n\nWITH LORA:\n{tuned}\n{'-' * 80}")

In [ ]:
adapter = {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}
torch.save(adapter, "adapter.pt")
size_mb = sum(p.numel() * p.element_size() for p in adapter.values()) / 1e6
print(
    f"OK: adapter.pt is {size_mb:.1f} MB; the full model is {n_total * 4 / 1e9:.1f} GB"
)

## Bonus: the same thing with PEFT (given)

In real projects you use the [PEFT](https://github.com/huggingface/peft) library. Inside, it does what you just wrote: `lora_A` and `lora_B` are `nn.Linear` layers without bias, and `lora_B` starts at zero. Check that it trains exactly as many parameters as yours.


In [ ]:
from peft import LoraConfig, get_peft_model

fresh = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32)
config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)
peft_model = get_peft_model(fresh, config)
n_peft = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
print(f"PEFT: {n_peft:,} trainable, yours: {n_trainable:,}")
print("OK: same count" if n_peft == n_trainable else "Different counts: check add_lora")